# PCA from Scratch: Pokémon Image Processing and Classification
### Beginner-friendly teaching notebook for the Machine Learning presentation

**This notebook is built around the professor's checklist** and explains the important formula/code lines step by step.

We will:
1. load and standardise Pokémon images;
2. compare an original image with controlled variations and explain why image processing/data augmentation can help;
3. implement PCA using NumPy: centering, covariance, eigen-decomposition, projection, explained variance and reconstruction;
4. visualise images in the first two principal components;
5. classify Pokémon using a separate nearest-centroid classifier;
6. compare our implementation with scikit-learn PCA, including a side-by-side image reconstruction;
7. compare PCA with t-SNE for visualisation only;
8. discuss linear vs. nonlinear dimensionality reduction;
9. show how the same PCA steps apply to high-dimensional cybersecurity features.

**Important:** PCA is not a classifier. PCA compresses/re-represents the images. The nearest-centroid algorithm is the classifier.

## Folder setup

Keep this notebook in the same folder as `pokemon_images`:

```text
PCA/
├── PCA_Pokemon_Teaching_Notebook_Final.ipynb
└── pokemon_images/
    ├── bulbasaur/
    │   ├── bulbasaur_01.jpeg
    │   ├── bulbasaur_02.jpeg
    │   └── bulbasaur_03.jpeg
    ├── charmander/
    │   ├── charmander_01.jpeg
    │   └── ...
    └── pikachu/
        ├── pikachu_01.jpeg
        └── ...
```

The notebook searches inside each class folder. Supported image formats are PNG, JPG, JPEG and BMP.

**Recommended runtime:** Python 3.11 or 3.12 with NumPy, pandas, Matplotlib, scikit-learn and ipykernel. The core PCA implementation uses NumPy; scikit-learn is used only for the requested comparison and t-SNE visualisation.

## 0. Quick environment check

Run this cell first. If it works but VS Code keeps showing “Connecting to kernel”, the notebook's Python kernel connection is the problem, not the PCA code.

In VS Code, select a Python kernel that can import these packages. The notebook uses small 16 × 16 RGB model inputs to keep the matrix calculations manageable.

In [ ]:
import sys
print("Python executable:", sys.executable)
print("Python version:", sys.version)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("Matplotlib:", plt.matplotlib.__version__)
print("Core imports successful.")

## 1. Set up the dataset

Each subfolder name becomes the class label. `IMAGE_HEIGHT` and `IMAGE_WIDTH` define the standard size used by the model.

**Why standardise image size?** A matrix must have a consistent number of columns. If one image has 300 × 300 pixels and another has 500 × 500, their flattened vectors have different lengths and cannot form one rectangular data matrix.

We use 16 × 16 RGB inputs for a quick demo: 16 × 16 × 3 = 768 pixel features per image.

In [ ]:
DATA_DIR = Path("pokemon_images")
IMAGE_HEIGHT = 16
IMAGE_WIDTH = 16
USE_COLOR = True

print("Dataset folder:", DATA_DIR.resolve())
print("Folder exists:", DATA_DIR.exists())

image_paths = sorted(
    p for p in DATA_DIR.rglob("*")
    if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp"}
) if DATA_DIR.exists() else []

print("Image files found:", len(image_paths))
print("Class folders found:", sorted({p.parent.name.lower() for p in image_paths}))

## 2. Load and standardise images

This cell uses Matplotlib to read the image and NumPy to resize it with a simple nearest-neighbour method. Values are scaled into the range 0 to 1.

Notice that **the source image and the model input are not the same thing**. The source may be high resolution; the model input is deliberately small to make PCA practical for a classroom demo.

In [ ]:
def resize_nearest(image, new_height=16, new_width=16):
    """Resize by selecting nearby existing pixels; no extra library required."""
    rows = np.linspace(0, image.shape[0] - 1, new_height).astype(int)
    cols = np.linspace(0, image.shape[1] - 1, new_width).astype(int)
    return image[rows][:, cols]

def load_image(path):
    image = plt.imread(path)

    # Remove alpha/transparency if present.
    if image.ndim == 3 and image.shape[2] >= 3:
        image = image[:, :, :3]

    # Optional grayscale mode.
    if image.ndim == 3 and not USE_COLOR:
        image = image.mean(axis=2)

    image = resize_nearest(image, IMAGE_HEIGHT, IMAGE_WIDTH).astype(np.float64)

    # PNG can already be 0..1; JPEG is often 0..255.
    if image.max() > 1:
        image = image / 255.0

    return np.clip(image, 0, 1)

def image_to_vector(image):
    """Flatten an image array into one long list of pixel features."""
    return image.reshape(-1)

if image_paths:
    source = plt.imread(image_paths[0])
    if source.ndim == 3 and source.shape[2] >= 3:
        source = source[:, :, :3]
    model_input = load_image(image_paths[0])

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(source, cmap="gray" if source.ndim == 2 else None)
    axes[0].set_title(f"Original file: {image_paths[0].parent.name}")
    axes[1].imshow(model_input, cmap="gray" if model_input.ndim == 2 else None)
    axes[1].set_title(f"Model input: {IMAGE_WIDTH} × {IMAGE_HEIGHT}")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    print("Source image shape:", source.shape)
    print("Model input shape:", model_input.shape)
    print("Pixel features:", image_to_vector(model_input).size)
else:
    print("No images found. Check that pokemon_images is beside this notebook and contains class subfolders.")

## 3. Original vs. changed images: why do image processing and data augmentation?

In real use, the same Pokémon can appear a little different because of position, brightness, scale, pose, or background. A model trained on only one exact image may learn those accidental details rather than features that generalise.

We will create **controlled, small variations** of each original image:
- shift one pixel to the right;
- shift one pixel down;
- make it slightly darker;
- make it slightly brighter.

These are examples of **data augmentation**. They give us a way to discuss how pixel features change when an image changes slightly. They can improve classification when the changes are realistic and the model sees them during training.

**Limitations:** synthetic variants are not new real photographs. Brightness changes and shifts do not teach every possible pose or background. We will keep original-image identity so that variants from the same source cannot accidentally appear in both training and test sets.

In [ ]:
def shift_image(image, dy=0, dx=0):
    """Shift without wrapping pixels around the opposite edge."""
    shifted = np.zeros_like(image)
    h, w = image.shape[:2]
    sy1, sy2 = max(0, -dy), min(h, h - dy)
    sx1, sx2 = max(0, -dx), min(w, w - dx)
    ty1, ty2 = max(0, dy), min(h, h + dy)
    tx1, tx2 = max(0, dx), min(w, w + dx)
    shifted[ty1:ty2, tx1:tx2] = image[sy1:sy2, sx1:sx2]
    return shifted

def make_variants(image):
    return [
        ("original", image),
        ("shift_right", shift_image(image, dx=1)),
        ("shift_down", shift_image(image, dy=1)),
        ("darker", np.clip(image * 0.85, 0, 1)),
        ("brighter", np.clip(image * 1.10, 0, 1)),
    ]

if image_paths:
    example = load_image(image_paths[0])
    variants = make_variants(example)
    fig, axes = plt.subplots(1, len(variants), figsize=(14, 3))
    for ax, (name, variant) in zip(axes, variants):
        ax.imshow(variant, cmap="gray" if variant.ndim == 2 else None)
        ax.set_title(name.replace("_", "\n"))
        ax.axis("off")
    plt.suptitle("One source image, five pixel-level versions")
    plt.tight_layout()
    plt.show()
else:
    print("Add image files before running this cell.")

### How can variations help classification?

The original and changed images have different pixel values, even though they represent the same class. If training includes reasonable variations, a later classifier may be less dependent on the exact pixel arrangement of a single image.

But **PCA itself does not learn invariance**. PCA simply finds directions of high variance. If brightness or shifts create a lot of variation, PCA may spend components describing those changes. Augmentation can help a classifier only when combined with a sound training and evaluation procedure.

## 4. Build the data matrix without train/test leakage

Each row of `X_original` is one original image flattened into a vector. We keep `source_id` so we know which file each row came from.

We will split the **original files first**. Then we augment training images only. Test images remain unmodified originals. This avoids a common evaluation mistake: putting one version of an image in training and a near-identical version of the same image in testing, which can make accuracy look artificially high.

In [ ]:
records = []
for path in image_paths:
    try:
        img = load_image(path)
        records.append({
            "path": str(path),
            "label": path.parent.name.lower(),
            "source_id": str(path),
            "image": img,
            "vector": image_to_vector(img)
        })
    except Exception as error:
        print(f"Skipping {path}: {error}")

if records:
    image_table = pd.DataFrame([{
        "path": r["path"], "label": r["label"], "source_id": r["source_id"]
    } for r in records])
    X_original = np.vstack([r["vector"] for r in records])
    y_original = np.array([r["label"] for r in records])
    print("Original-image matrix shape:", X_original.shape)
    display(image_table.head())
    display(image_table["label"].value_counts().rename("images_per_class").to_frame())
else:
    X_original = np.empty((0, IMAGE_HEIGHT * IMAGE_WIDTH * (3 if USE_COLOR else 1)))
    y_original = np.array([])
    print("No images loaded.")

## 5. Split original files into training and test examples

For each class, we use all but the last original file for training and hold out the last original file for testing, if the class has at least two images. This is a tiny classroom split, not a robust research evaluation.

If you have only one original image in a class, it cannot be used to both train and test that class. Add more varied real images for a meaningful result.

In [ ]:
train_record_indices, test_record_indices = [], []
if records:
    for label in sorted(set(y_original)):
        ids = np.where(y_original == label)[0]
        if len(ids) >= 2:
            train_record_indices.extend(ids[:-1])
            test_record_indices.append(ids[-1])

train_record_indices = np.array(train_record_indices, dtype=int)
test_record_indices = np.array(test_record_indices, dtype=int)

print("Original files for training:", len(train_record_indices))
print("Held-out original files for testing:", len(test_record_indices))
print("Training classes:", sorted(set(y_original[train_record_indices])) if len(train_record_indices) else [])

## 6. Augment training images only and create X_train

The PCA directions are learned from the training set only. This avoids using test data to decide the representation.

For a compact classroom implementation, we create the variants from training originals and fit PCA on those augmented training vectors. The test vectors are transformed using the **training mean and training principal components**.

In [ ]:
train_vectors, train_labels = [], []
for idx in train_record_indices:
    image = records[idx]["image"]
    label = records[idx]["label"]
    for variant_name, variant in make_variants(image):
        train_vectors.append(image_to_vector(variant))
        train_labels.append(label)

X_train = np.vstack(train_vectors) if train_vectors else np.empty((0, X_original.shape[1]))
y_train = np.array(train_labels)
X_test = X_original[test_record_indices] if len(test_record_indices) else np.empty((0, X_original.shape[1]))
y_test = y_original[test_record_indices] if len(test_record_indices) else np.array([])

print("Augmented training matrix:", X_train.shape)
print("Unmodified test matrix:", X_test.shape)

## 7. PCA step 1: centre each pixel feature

Formula:

\[
X_c = X_{train} - \mu
\]

`axis=0` means “calculate one mean per column” across training images. We subtract that mean from each image. PCA looks for variation around the mean, so centering is an important first step.

In [ ]:
if len(X_train) >= 2:
    mean_image = np.mean(X_train, axis=0)
    X_centered = X_train - mean_image
    print("Original training means, first 5:", np.mean(X_train, axis=0)[:5])
    print("Centred means, first 5:", np.mean(X_centered, axis=0)[:5])
else:
    mean_image = None
    X_centered = None
    print("Need at least two training image versions.")

## 8. PCA step 2: calculate the covariance matrix

\[
C = \frac{X_c^T X_c}{n-1}
\]

The covariance matrix describes how pairs of pixel features vary together. If there are `p` features, `C` has shape `p × p`. With 16 × 16 RGB inputs, `p = 768`, so the matrix is 768 × 768.

We use the sample covariance formula, dividing by `n - 1`, where `n` is the number of training rows.

In [ ]:
if X_centered is not None:
    n_train, n_features = X_centered.shape
    covariance = (X_centered.T @ X_centered) / (n_train - 1)
    print("Training rows n:", n_train)
    print("Pixel features p:", n_features)
    print("Covariance matrix shape:", covariance.shape)
else:
    covariance = None

## 9. PCA step 3: eigen-decomposition gives principal components

For covariance matrix `C`, an eigenvector `v` and eigenvalue `λ` satisfy:

\[
Cv = \lambda v
\]

- Eigenvectors are the principal-component directions.
- Eigenvalues measure the variance along those directions.
- The largest eigenvalue corresponds to the first principal component.

`np.linalg.eigh` is appropriate for a symmetric covariance matrix. It returns eigenvalues in ascending order, so we sort them from largest to smallest.

In [ ]:
if covariance is not None:
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = np.maximum(eigenvalues[order], 0)
    eigenvectors = eigenvectors[:, order]
    print("Largest five eigenvalues:", eigenvalues[:5])
    print("Eigenvector matrix shape:", eigenvectors.shape)
else:
    eigenvalues = eigenvectors = None

## 10. Explained variance ratio and scree plot

\[
r_i = \frac{\lambda_i}{\sum_j \lambda_j}
\]

This ratio tells us what fraction of the total variance is associated with component `i`. Cumulative explained variance sums the first `k` ratios. A scree plot helps show where additional components contribute less.

**Choosing k is a trade-off:** fewer components give stronger compression but may lose details. The component count that preserves variance best is not necessarily the component count that gives the best classification accuracy.

In [ ]:
if eigenvalues is not None and eigenvalues.sum() > 0:
    explained_variance_ratio = eigenvalues / eigenvalues.sum()
    cumulative_variance = np.cumsum(explained_variance_ratio)

    variance_table = pd.DataFrame({
        "component": np.arange(1, len(eigenvalues) + 1),
        "explained_variance_ratio": explained_variance_ratio,
        "cumulative_variance": cumulative_variance
    })
    display(variance_table.head(10))

    plt.figure(figsize=(8, 4))
    plt.plot(np.arange(1, len(eigenvalues) + 1), explained_variance_ratio, marker=".")
    plt.xlabel("Principal component")
    plt.ylabel("Explained variance ratio")
    plt.title("Scree plot")
    plt.grid(True, alpha=0.3)
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.plot(np.arange(1, len(eigenvalues) + 1), cumulative_variance)
    plt.axhline(0.90, linestyle="--", label="90% variance")
    plt.xlabel("Number of components k")
    plt.ylabel("Cumulative explained variance")
    plt.title("Variance retained by the first k components")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    explained_variance_ratio = cumulative_variance = None
    print("No usable variance found yet.")

## 11. Project into the top-k principal components

The first `k` eigenvectors form the projection matrix `W_k`.

\[
Z = X_c W_k
\]

The reduced representation `Z` contains fewer numbers per image. We choose `k` conservatively so the example remains understandable and runs quickly.

In [ ]:
if eigenvectors is not None:
    k = min(20, len(eigenvalues), len(X_train) - 1)
    W_k = eigenvectors[:, :k]
    Z_train = X_centered @ W_k
    X_test_centered = X_test - mean_image
    Z_test = X_test_centered @ W_k if len(X_test) else np.empty((0, k))
    print("Chosen k:", k)
    print("Original training shape:", X_train.shape)
    print("Reduced training shape:", Z_train.shape)
    print("Reduced test shape:", Z_test.shape)
else:
    k = 0
    W_k = Z_train = Z_test = None

## 12. PCA graph: each image version becomes a point

We plot the first two principal components. Each dot is an augmented **training image version**, coloured by its folder label.

- PC1 and PC2 are directions of high variance, not class names.
- Clusters can overlap because PCA does not explicitly optimise class separation.
- If points spread out, that can reflect position or brightness changes, not just Pokémon identity.
- The axis percentages show the variance captured by each plotted component.

This is a visual explanation of the reduced space, not proof of model accuracy.

In [ ]:
if Z_train is not None and Z_train.shape[1] >= 2:
    plot_df = pd.DataFrame({"PC1": Z_train[:, 0], "PC2": Z_train[:, 1], "class": y_train})
    plt.figure(figsize=(9, 6))
    for label in sorted(plot_df["class"].unique()):
        group = plot_df[plot_df["class"] == label]
        plt.scatter(group["PC1"], group["PC2"], label=label.title(),
                    alpha=0.72, s=48, edgecolors="black", linewidths=0.3)
    plt.xlabel(f"PC1 ({explained_variance_ratio[0]*100:.1f}% variance)")
    plt.ylabel(f"PC2 ({explained_variance_ratio[1]*100:.1f}% variance)")
    plt.title("Training image variations in PCA space")
    plt.legend(title="Pokémon class")
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print("Need at least two PCA components to plot.")

## 13. Reconstruct an image and compare it with the input

The approximate inverse mapping is:

\[
\hat{X} = ZW_k^T + \mu
\]

The original model input is already resized to 16 × 16, so it looks pixelated compared with a high-resolution source. The reconstruction shows the **additional** loss caused by keeping only `k` components.

We compare the original model input with the reconstruction and calculate mean squared error (MSE), the average squared pixel difference.

In [ ]:
if W_k is not None and len(X_train):
    train_index = 0
    reconstructed_train = Z_train @ W_k.T + mean_image
    original_model_image = X_train[train_index].reshape(IMAGE_HEIGHT, IMAGE_WIDTH, 3) if USE_COLOR else X_train[train_index].reshape(IMAGE_HEIGHT, IMAGE_WIDTH)
    reconstructed_image = np.clip(reconstructed_train[train_index], 0, 1)
    reconstructed_image = reconstructed_image.reshape(original_model_image.shape)

    fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
    axes[0].imshow(original_model_image, cmap="gray" if original_model_image.ndim == 2 else None)
    axes[0].set_title("Original model input")
    axes[1].imshow(reconstructed_image, cmap="gray" if reconstructed_image.ndim == 2 else None)
    axes[1].set_title(f"NumPy PCA reconstruction (k={k})")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    mse = np.mean((X_train[train_index] - reconstructed_train[train_index]) ** 2)
    print("Reconstruction MSE:", round(float(mse), 6))

## 14. Classification is a separate step: nearest centroid

For each class, compute the mean point (centroid) in PCA space. For a test image, predict the label of the closest centroid.

\[
d(z,c) = \sqrt{\sum_j (z_j-c_j)^2}
\]

The PCA algorithm does not use labels to find components. The classifier uses labels afterwards to create class centroids. We test on original files held out before augmentation.

In [ ]:
def fit_centroids(train_Z, train_y):
    return {label: train_Z[train_y == label].mean(axis=0)
            for label in np.unique(train_y)}

def predict_centroid(test_Z, centroids):
    predictions = []
    for row in test_Z:
        distances = {label: np.linalg.norm(row - center)
                     for label, center in centroids.items()}
        predictions.append(min(distances, key=distances.get))
    return np.array(predictions)

if Z_train is not None and len(Z_test) and len(set(y_train)):
    centroids = fit_centroids(Z_train, y_train)
    predictions = predict_centroid(Z_test, centroids)
    results = pd.DataFrame({
        "test_image": [records[i]["path"] for i in test_record_indices],
        "actual": y_test,
        "predicted": predictions,
        "correct": predictions == y_test
    })
    display(results)
    print("Tiny holdout accuracy:", f"{np.mean(predictions == y_test):.1%}")
    print("Caution: with only a few original images, this score is highly uncertain.")
else:
    centroids = {}
    print("Need at least two original images per class to create a held-out example.")

## 15. Optional unknown-class rejection

A nearest-centroid classifier always chooses the closest known class. For a simple “unknown” option, we can reject a prediction when the nearest centroid is farther away than a threshold.

The threshold is **not automatically a probability or confidence score**. It should be selected using validation data. A new class also needs labelled examples before the model can build a centroid for it.

In [ ]:
def predict_with_unknown(test_point, centroids, max_distance):
    distances = {label: np.linalg.norm(test_point - center)
                 for label, center in centroids.items()}
    if not distances:
        return "Unknown", float("inf")
    label = min(distances, key=distances.get)
    distance = float(distances[label])
    return (label if distance <= max_distance else "Unknown"), distance

print("After validating a suitable threshold, call predict_with_unknown(Z_test[0], centroids, max_distance=YOUR_THRESHOLD).")

## 16. scikit-learn PCA comparison: numerical and visual evidence

This is the requested library comparison. The main PCA implementation above is written explicitly using NumPy. Here we use `sklearn.decomposition.PCA` as a reference.

We compare:
- cumulative explained variance;
- reconstruction MSE;
- the same model input reconstructed by our implementation and by scikit-learn.

The two reconstructed images should be broadly similar. Small differences are normal because eigenvectors can have different signs while representing the same principal directions.

In [ ]:
if X_centered is not None and k >= 1:
    try:
        from sklearn.decomposition import PCA

        sklearn_pca = PCA(n_components=k)
        sklearn_Z_train = sklearn_pca.fit_transform(X_train)
        sklearn_reconstructed = sklearn_pca.inverse_transform(sklearn_Z_train)
        numpy_reconstructed = Z_train @ W_k.T + mean_image

        comparison = pd.DataFrame({
            "method": ["NumPy PCA from scratch", "scikit-learn PCA"],
            "k": [k, k],
            "cumulative_explained_variance": [
                float(explained_variance_ratio[:k].sum()),
                float(sklearn_pca.explained_variance_ratio_.sum())
            ],
            "reconstruction_MSE": [
                float(np.mean((X_train - numpy_reconstructed) ** 2)),
                float(np.mean((X_train - sklearn_reconstructed) ** 2))
            ]
        })
        display(comparison)

        index = 0
        shape = (IMAGE_HEIGHT, IMAGE_WIDTH, 3) if USE_COLOR else (IMAGE_HEIGHT, IMAGE_WIDTH)
        original_display = X_train[index].reshape(shape)
        numpy_display = np.clip(numpy_reconstructed[index].reshape(shape), 0, 1)
        sklearn_display = np.clip(sklearn_reconstructed[index].reshape(shape), 0, 1)

        fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
        axes[0].imshow(original_display, cmap="gray" if original_display.ndim == 2 else None)
        axes[0].set_title("Original model input")
        axes[1].imshow(numpy_display, cmap="gray" if numpy_display.ndim == 2 else None)
        axes[1].set_title(f"NumPy PCA (k={k})")
        axes[2].imshow(sklearn_display, cmap="gray" if sklearn_display.ndim == 2 else None)
        axes[2].set_title(f"scikit-learn PCA (k={k})")
        for ax in axes:
            ax.axis("off")
        plt.tight_layout()
        plt.show()
    except ImportError:
        print("Install scikit-learn in the terminal: python -m pip install scikit-learn")
else:
    print("Run PCA cells first.")

## 17. Alternative PCA route: Singular Value Decomposition (SVD)

For centred data, SVD writes:

\[
X_c = U\Sigma V^T
\]

The columns of `V` are principal-component directions. The covariance eigenvalues are related to singular values by:

\[
\lambda_i = \frac{\sigma_i^2}{n-1}
\]

SVD can be more practical than explicitly creating a very large covariance matrix. This cell demonstrates the NumPy alternative, while the covariance/eigen route remains the main teaching implementation.

In [ ]:
if X_centered is not None:
    U, singular_values, Vt = np.linalg.svd(X_centered, full_matrices=False)
    svd_variances = singular_values ** 2 / (len(X_centered) - 1)
    svd_ratios = svd_variances / svd_variances.sum()
    print("SVD directions shape:", Vt.shape)
    print("First five SVD variance ratios:", np.round(svd_ratios[:5], 4))
    print("First five covariance-eigen ratios:", np.round(explained_variance_ratio[:5], 4))

## 18. PCA versus t-SNE: nonlinear visualisation only

PCA is **linear**: each component is a weighted sum of the original features and is chosen to preserve variance.

t-SNE is **nonlinear**: it tries to keep similar local neighbourhoods close together in a 2D visualisation. Its axes do not represent variance percentages, and distances between far-apart groups can be misleading.

We use t-SNE only to visualise the same training representations. It is not a classifier and does not replace the PCA algorithm.

In [ ]:
if Z_train is not None and len(Z_train) >= 5:
    try:
        from sklearn.manifold import TSNE
        perplexity_value = min(5.0, max(2.0, (len(Z_train) - 1) / 3))
        tsne = TSNE(n_components=2, perplexity=perplexity_value,
                    init="pca", learning_rate="auto", random_state=42)
        Z_tsne = tsne.fit_transform(Z_train)

        plt.figure(figsize=(9, 6))
        for label in sorted(np.unique(y_train)):
            mask = y_train == label
            plt.scatter(Z_tsne[mask, 0], Z_tsne[mask, 1],
                        label=label.title(), alpha=0.72, s=45)
        plt.title("t-SNE view of PCA training representations")
        plt.xlabel("t-SNE dimension 1 (not variance)")
        plt.ylabel("t-SNE dimension 2 (not variance)")
        plt.legend(title="Pokémon class")
        plt.grid(True, alpha=0.2)
        plt.tight_layout()
        plt.show()
    except ImportError:
        print("Install scikit-learn to run t-SNE.")
else:
    print("Need at least five training representations for t-SNE.")

### When might PCA fail while a nonlinear method helps?

If the data's important structure lies along a curved or complicated shape, a straight-line projection may not represent it well in two dimensions. A nonlinear visualisation such as t-SNE may reveal local groups that PCA's plot hides.

However, a visually separated t-SNE plot does not guarantee better classification. Always evaluate the classifier on held-out examples. For the Pokémon task, consistent image framing, more real training images, and careful evaluation may matter more than changing the visualisation method.

## 19. Cybersecurity application: PCA on high-dimensional network features

A network flow can be described by many numeric features, such as duration, bytes sent/received, packet counts, average packet size, connection rates, and protocol indicators. PCA can reduce these features for exploration or as an input to a later model.

The example below creates **synthetic network-flow-style features** so it runs without downloading a dataset. The labels are only used to colour the plot. PCA itself does not use the labels.

PCA is not an intrusion detector: high variance does not automatically mean malicious traffic, and low-variance features can still matter for security. Real work needs real data, careful train/test splits, and suitable evaluation metrics.

In [ ]:
rng = np.random.default_rng(42)
n_normal, n_suspicious = 80, 25
normal_flows = rng.normal(0.0, 1.0, size=(n_normal, 8))
suspicious_flows = rng.normal(0.8, 1.35, size=(n_suspicious, 8))
network_X = np.vstack([normal_flows, suspicious_flows])
network_labels = np.array(["normal"] * n_normal + ["synthetic suspicious example"] * n_suspicious)

network_mean = network_X.mean(axis=0)
network_centered = network_X - network_mean
network_cov = network_centered.T @ network_centered / (len(network_X) - 1)
net_values, net_vectors = np.linalg.eigh(network_cov)
net_order = np.argsort(net_values)[::-1]
net_values = np.maximum(net_values[net_order], 0)
net_vectors = net_vectors[:, net_order]
network_Z = network_centered @ net_vectors[:, :2]
network_ratios = net_values / net_values.sum()

plt.figure(figsize=(8, 5))
for label in sorted(np.unique(network_labels)):
    mask = network_labels == label
    plt.scatter(network_Z[mask, 0], network_Z[mask, 1], label=label, alpha=0.7)
plt.xlabel(f"PC1 ({network_ratios[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({network_ratios[1]*100:.1f}% variance)")
plt.title("PCA on synthetic network-flow features")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

print("Synthetic network feature matrix:", network_X.shape)
print(f"Variance retained by two components: {network_ratios[:2].sum()*100:.1f}%")
print("This is an educational simulation, not a real security-detection result.")

## 20. Final checklist for the professor's instructions

- [x] NumPy-based PCA main implementation
- [x] Centre data and compute covariance matrix
- [x] Eigen-decomposition to extract principal components
- [x] Project data onto top-k components
- [x] Explained variance ratio and scree plot
- [x] Reconstruction from reduced representation
- [x] Comparison with `sklearn.decomposition.PCA`, including reconstructed-image output
- [x] Comparison with t-SNE using `sklearn.manifold.TSNE` for visualisation only
- [x] Discussion of linear vs. nonlinear dimensionality reduction and when PCA may fail
- [x] Cybersecurity application using high-dimensional network-style features

### Suggested split between two presenters

| Presenter | Sections | Key teaching task |
|---|---|---|
| Presenter A | 1–8 | Image pixels, standardisation, variants, matrix, centering, covariance |
| Presenter B | 9–13 | Eigenvectors/eigenvalues, variance, projection, graph, reconstruction |
| Both | 14–20 | Classification, scikit-learn comparison, SVD, t-SNE, cybersecurity, limitations |

### The five lines worth live-coding

1. `X_centered = X_train - mean_image`
2. `covariance = (X_centered.T @ X_centered) / (n_train - 1)`
3. Sorting eigenvalues from largest to smallest
4. `Z_train = X_centered @ W_k`
5. `reconstructed_train = Z_train @ W_k.T + mean_image`

**Final takeaway:** PCA compresses features by preserving variance. It does not automatically preserve class identity or classify Pokémon. Image standardisation and sensible augmentation can help a separate classifier generalise, but real performance must be measured on held-out original images.